# 02 - Block-wise width (PFOR)

An outlier inflates only its own block. Here we measure the memory of the block-wise width vs the safe single_w width, varying the outlier fraction. Uses the real `S2RBlocked` type.

In [ ]:
import os, subprocess, tempfile
import matplotlib.pyplot as plt

def find_include():
    d = os.getcwd()
    for _ in range(6):
        cand = os.path.join(d, "include", "smart2raw.h")
        if os.path.exists(cand):
            return os.path.join(d, "include")
        d = os.path.dirname(d)
    raise FileNotFoundError("include/smart2raw.h not found; run from the repo")

INC = find_include()

def compile_run(src, args_list, flags="-O3 -march=native"):
    """Compiles a C harness (with the header) and runs it for each args; returns outputs."""
    with tempfile.NamedTemporaryFile("w", suffix=".c", delete=False) as f:
        f.write(src); cpath = f.name
    exe = cpath[:-2]
    subprocess.check_call(["gcc"] + flags.split() + ["-I", INC, "-o", exe, cpath])
    outs = []
    for a in args_list:
        outs.append(subprocess.check_output([exe] + [str(x) for x in a]).decode().strip())
    os.remove(cpath); os.remove(exe)
    return outs

print("include:", INC)

## Measurement

In [ ]:
SRC = r"""#include <stdio.h>
#include <stdlib.h>
#include "smart2raw.h"
int main(int argc,char**argv){
  long ppm=argc>1?atol(argv[1]):100; size_t N=4000000; srand(1);
  uint64_t *x=malloc(N*sizeof(uint64_t));
  for(size_t i=0;i<N;i++) x[i]=(uint64_t)(i%200);
  size_t k=(size_t)((double)N*ppm/1e6);
  for(size_t j=0;j<k;j++) x[(size_t)rand()%N]=5000000+rand()%1000000;
  S2RBlocked b; s2r_blocked_build(&b,x,N,256);
  uint64_t mx=0; for(size_t i=0;i<N;i++) if(x[i]>mx)mx=x[i];
  size_t single_w=(size_t)(s2r_classify(mx)/8)*N;
  printf("%ld %zu %zu\n",ppm,single_w,s2r_blocked_bytes(&b));
  s2r_blocked_free(&b); free(x); return 0;}"""

ppms = [0,10,50,100,200,500,1000,2000,5000,10000]   # parts per million of outliers
rows = [list(map(float, o.split())) for o in compile_run(SRC, [[p] for p in ppms])]
ppm = [r[0] for r in rows]
ratio = [ (r[1]/r[2]) if r[2] else 1.0 for r in rows ]  # single width / block-wise
for p,r in zip(ppm,ratio): print(f"{p/1e4:5.2f}% outliers  ->  {r:4.2f}x memory recovered")

## Chart

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
ax.plot([p/1e4 for p in ppm], ratio, "o-", color="#2E7D5B", lw=2)
ax.axhline(1.0, color="#888", ls=":")
ax.set_xlabel("% outliers"); ax.set_ylabel("memory: single_w width / per block")
ax.set_title("PFOR: memory recovery vs outlier fraction (measured)")
ax.grid(alpha=0.3); plt.show()

Expected: ~3.7x when outliers are rare (0.01%) and a gradual drop as they multiply (more blocks need the wide class). On clean data, ~1x (no harm).